# Qa 04 bathy patches

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# QA 04: Bathymetry Patches

Validate bathymetry patch shapes, channels, normalization metadata, and split-aware examples for both bathy v1 and bathy v2 artifacts.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks.multisource_notebook_helpers import (
    read_yaml,
    resolve_point_centric_dir,
    load_metadata,
    load_bathy_artifact,
)

try:
    import contextily as ctx
except Exception:
    ctx = None

try:
    from pyproj import Transformer
except Exception:
    Transformer = None

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
CONFIG_PATH = Path("../configs/training.yaml")
POINT_CENTRIC_DIR = resolve_point_centric_dir(CONFIG_PATH)
PROBLEM_SITES = []  # e.g. ['norac_grid_93']
RANDOM_SEED = 45
N_RANDOM_SITES = 3

In [ ]:
metadata = load_metadata(POINT_CENTRIC_DIR)
bathy_npz = load_bathy_artifact(POINT_CENTRIC_DIR)
x_bathy = bathy_npz["X_bathy"]
channel_names = (
    bathy_npz["channel_names"].astype(str).tolist() if "channel_names" in bathy_npz else []
)
target_sites = bathy_npz["target_sites"].astype(str).tolist() if "target_sites" in bathy_npz else []
patch_size = (
    int(np.asarray(bathy_npz["patch_size"]).reshape(-1)[0])
    if "patch_size" in bathy_npz
    else x_bathy.shape[-1]
)
resolution_m = (
    float(np.asarray(bathy_npz["resolution_m"]).reshape(-1)[0])
    if "resolution_m" in bathy_npz
    else np.nan
)
norm_meta = (
    bathy_npz["normalization_metadata"].item() if "normalization_metadata" in bathy_npz else {}
)
print("X_bathy shape:", x_bathy.shape)
print("channel names:", channel_names)
print("patch_size:", patch_size)
print("resolution_m:", resolution_m)
print("normalization metadata keys:", list((norm_meta or {}).keys()))

In [ ]:
summary_rows = []
for idx, name in enumerate(channel_names):
    arr = np.asarray(x_bathy[:, idx, :, :], dtype=float)
    finite = arr[np.isfinite(arr)]
    summary_rows.append(
        {
            "channel": name,
            "nan_count": int(np.isnan(arr).sum()),
            "min": float(np.nanmin(arr)) if finite.size else np.nan,
            "median": float(np.nanmedian(arr)) if finite.size else np.nan,
            "max": float(np.nanmax(arr)) if finite.size else np.nan,
            "is_constant": bool(finite.size and np.allclose(finite, finite[0])),
        }
    )
summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
cfg = read_yaml(CONFIG_PATH)
model_bathy_cfg = ((cfg.get("model", {}) or {}).get("coastal_transformer", {}) or {}).get(
    "bathy", {}
) or {}
expected_channels = int(model_bathy_cfg.get("in_channels", x_bathy.shape[1]))
expected_patch_size = int(model_bathy_cfg.get("patch_size", x_bathy.shape[-1]))
if expected_channels != x_bathy.shape[1]:
    print(f"WARNING: config in_channels={expected_channels} but data has {x_bathy.shape[1]}")
if expected_patch_size != x_bathy.shape[-1]:
    print(f"WARNING: config patch_size={expected_patch_size} but data has {x_bathy.shape[-1]}")
if any(summary_df["is_constant"]):
    print("WARNING: at least one bathy channel is constant")
mask_idx = next(
    (i for i, name in enumerate(channel_names) if name in {"wet_mask", "land_sea_mask"}), None
)
if mask_idx is not None:
    mask_mean = float(np.nanmean(x_bathy[:, mask_idx, :, :]))
    print("Mask mean:", mask_mean)
    if mask_mean < 0.1 or mask_mean > 0.9:
        print("WARNING: land/sea mask may be inverted or heavily imbalanced")

In [ ]:
rng = random.Random(RANDOM_SEED)
split_meta = metadata.get("splits", {}) or {}
site_pool = sorted(
    set(PROBLEM_SITES) | set(rng.sample(target_sites, min(N_RANDOM_SITES, len(target_sites))))
)
lookup = {site: idx for idx, site in enumerate(target_sites)}
for site in site_pool:
    if site not in lookup:
        print("Skipping missing site:", site)
        continue
    patch = x_bathy[lookup[site]]
    fig, axes = plt.subplots(1, patch.shape[0], figsize=(4 * patch.shape[0], 4))
    axes = np.atleast_1d(axes)
    for cidx, ax in enumerate(axes):
        im = ax.imshow(patch[cidx], cmap="viridis")
        ax.set_title(f"{site}\n{channel_names[cidx]}")
        ax.set_xticks([])
        ax.set_yticks([])
        plt.colorbar(im, ax=ax, shrink=0.75)
    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(len(channel_names), 1, figsize=(8, 3 * len(channel_names)))
axes = np.atleast_1d(axes)
for idx, name in enumerate(channel_names):
    vals = np.asarray(x_bathy[:, idx, :, :], dtype=float).ravel()
    vals = vals[np.isfinite(vals)]
    axes[idx].hist(vals, bins=60, color="#1f77b4", alpha=0.8)
    axes[idx].set_title(name)
plt.tight_layout()
plt.show()